# Categorical Missing Value Imputation

## Objective

In this notebook, we will:

- Preserve the original Heart Failure dataset
- Convert binary numerical columns into readable category labels
- Introduce categorical missing values in a practice copy
- Apply Mode Imputation
- Create a new `"Missing"` category
- Apply Most Frequent Imputation using `SimpleImputer`
- Apply Machine Learning-Based Imputation
- Compare the results
- Verify that the original dataset remains unchanged


In [1]:
# Import required libraries
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

In [3]:
# Load the original Heart Failure dataset
df = pd.read_csv(
    "../heart_failure_clinical_records_dataset-selected-columns.csv"
)

df.head()

,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex
0,75.0,0,582,0,20,1,265000.00,1.9,130,1
1,55.0,0,7861,0,38,0,263358.03,1.1,136,1
2,65.0,0,146,0,20,0,162000.00,1.3,129,1
3,50.0,1,111,0,20,0,210000.00,1.9,137,1
4,65.0,1,160,1,20,0,327000.00,2.7,116,0


In [4]:
# Verify the original dataset
print("Dataset Shape:", df.shape)

print(
    "Original Missing Values:",
    df.isnull().sum().sum()
)

Dataset Shape: (299, 10)
Original Missing Values: 0


## Create Readable Categorical Labels

The dataset stores binary categories as `0` and `1`.

For easier understanding:

- Diabetes: `0 → No`, `1 → Yes`
- Sex: `0 → Female`, `1 → Male`

The original columns will remain unchanged.

In [5]:
# Create an independent practice copy
df_categorical = df.copy()

# Convert binary values into readable labels
df_categorical["diabetes_label"] = (
    df_categorical["diabetes"].map({
        0: "No",
        1: "Yes"
    })
)

df_categorical["sex_label"] = (
    df_categorical["sex"].map({
        0: "Female",
        1: "Male"
    })
)

df_categorical[
    [
        "diabetes",
        "diabetes_label",
        "sex",
        "sex_label"
    ]
].head()

,diabetes,diabetes_label,sex,sex_label
0,0,No,1,Male
1,0,No,1,Male
2,0,No,1,Male
3,0,No,1,Male
4,1,Yes,0,Female


In [6]:
# Check category frequencies before adding missing values
print("Diabetes Categories:")
print(
    df_categorical[
        "diabetes_label"
    ].value_counts()
)

print("\nSex Categories:")
print(
    df_categorical[
        "sex_label"
    ].value_counts()
)

Diabetes Categories:
diabetes_label
No     174
Yes    125
Name: count, dtype: int64

Sex Categories:
sex_label
Male      194
Female    105
Name: count, dtype: int64


## Introduce Missing Values

Missing values will be introduced only in the practice label columns.

The original dataset and original binary columns will remain unchanged.

In [7]:
# Introduce missing categorical values

diabetes_missing_indexes = [
    5, 10, 15, 20, 25
]

sex_missing_indexes = [
    30, 35, 40, 45, 50
]

df_categorical.loc[
    diabetes_missing_indexes,
    "diabetes_label"
] = np.nan

df_categorical.loc[
    sex_missing_indexes,
    "sex_label"
] = np.nan

df_categorical[
    [
        "diabetes_label",
        "sex_label"
    ]
].isnull().sum()

diabetes_label    5
sex_label         5
dtype: int64

In [8]:
# Display rows containing missing category labels
df_categorical.loc[
    sorted(
        diabetes_missing_indexes
        + sex_missing_indexes
    ),
    [
        "age",
        "diabetes_label",
        "sex_label"
    ]
]

,age,diabetes_label,sex_label
5,90.0,NaN,Male
10,75.0,NaN,Male
15,82.0,NaN,Male
20,65.0,NaN,Female
25,80.0,NaN,Male
30,94.0,Yes,NaN
35,69.0,Yes,NaN
40,70.0,No,NaN
45,50.0,Yes,NaN
50,68.0,No,NaN


# Mode Imputation

Mode Imputation fills missing values with the most frequently occurring category.

In [9]:
# Calculate the mode of Diabetes Label
diabetes_mode = (
    df_categorical[
        "diabetes_label"
    ].mode()[0]
)

print("Diabetes Mode:", diabetes_mode)

Diabetes Mode: No


In [10]:
# Create a separate copy for Mode Imputation
df_mode = df_categorical.copy()

df_mode["diabetes_label"] = (
    df_mode["diabetes_label"]
    .fillna(diabetes_mode)
)

df_mode.loc[
    diabetes_missing_indexes,
    ["diabetes_label"]
]

,diabetes_label
5,No
10,No
15,No
20,No
25,No


In [11]:
print(
    "Remaining Missing Diabetes Labels:",
    df_mode["diabetes_label"]
    .isnull()
    .sum()
)

Remaining Missing Diabetes Labels: 0


# New Missing Category

This method preserves missingness by replacing unavailable values with a separate category called `"Missing"`.

In [12]:
# Create a separate copy
df_new_category = df_categorical.copy()

df_new_category["sex_label"] = (
    df_new_category["sex_label"]
    .fillna("Missing")
)

df_new_category.loc[
    sex_missing_indexes,
    ["sex_label"]
]

,sex_label
30,Missing
35,Missing
40,Missing
45,Missing
50,Missing


In [13]:
# Check updated category distribution
df_new_category[
    "sex_label"
].value_counts(dropna=False)

sex_label
Male       189
Female     105
Missing      5
Name: count, dtype: int64

# Most Frequent Imputation

`SimpleImputer` replaces missing categorical values with the most frequent category.

This implementation is suitable for Machine Learning pipelines.

In [14]:
# Create a separate copy
df_most_frequent = df_categorical.copy()

# Create Most Frequent imputer
most_frequent_imputer = SimpleImputer(
    strategy="most_frequent"
)

In [15]:
# Fit and transform the Diabetes Label column
df_most_frequent[
    ["diabetes_label"]
] = most_frequent_imputer.fit_transform(
    df_most_frequent[
        ["diabetes_label"]
    ]
)

df_most_frequent.loc[
    diabetes_missing_indexes,
    ["diabetes_label"]
]

,diabetes_label
5,No
10,No
15,No
20,No
25,No


In [16]:
print(
    "Remaining Missing Diabetes Labels:",
    df_most_frequent[
        "diabetes_label"
    ].isnull().sum()
)

print(
    "Learned Category:",
    most_frequent_imputer.statistics_[0]
)

Remaining Missing Diabetes Labels: 0
Learned Category: No


# Machine Learning-Based Imputation

A classification model will predict missing Diabetes categories using available numerical features.

## Target

`diabetes_label`

## Predictors

- Age
- Ejection Fraction
- Serum Creatinine
- Serum Sodium
- High Blood Pressure

In [17]:
# Create a separate copy for ML-based imputation
df_ml = df_categorical.copy()

target_column = "diabetes_label"

predictor_columns = [
    "age",
    "ejection_fraction",
    "serum_creatinine",
    "serum_sodium",
    "high_blood_pressure"
]

In [18]:
# Rows where target category is available
training_mask = (
    df_ml[target_column].notnull()
    & df_ml[predictor_columns]
    .notnull()
    .all(axis=1)
)

# Rows where target category is missing
prediction_mask = (
    df_ml[target_column].isnull()
    & df_ml[predictor_columns]
    .notnull()
    .all(axis=1)
)

print(
    "Training Rows:",
    training_mask.sum()
)

print(
    "Rows to Predict:",
    prediction_mask.sum()
)

Training Rows: 294
Rows to Predict: 5


In [19]:
# Prepare model training data
X_train = df_ml.loc[
    training_mask,
    predictor_columns
]

y_train = df_ml.loc[
    training_mask,
    target_column
]

X_missing = df_ml.loc[
    prediction_mask,
    predictor_columns
]

print("X Train Shape:", X_train.shape)
print("X Missing Shape:", X_missing.shape)

X Train Shape: (294, 5)
X Missing Shape: (5, 5)


In [20]:
# Create Random Forest classifier
categorical_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

# Train the model
categorical_model.fit(
    X_train,
    y_train
)

print("Classification Model Trained")

Classification Model Trained


In [21]:
# Predict missing Diabetes categories
predicted_categories = (
    categorical_model.predict(
        X_missing
    )
)

predicted_categories

array(['No', 'No', 'No', 'Yes', 'No'], dtype=object)

In [22]:
# Fill missing categories with predictions
df_ml.loc[
    prediction_mask,
    target_column
] = predicted_categories

df_ml.loc[
    prediction_mask,
    predictor_columns + [target_column]
]

,age,ejection_fraction,serum_creatinine,serum_sodium,high_blood_pressure,diabetes_label
5,90.0,40,2.1,132,1,No
10,75.0,38,4.0,131,1,No
15,82.0,50,1.3,136,0,No
20,65.0,25,1.3,137,1,Yes
25,80.0,38,1.9,144,0,No


In [23]:
print(
    "Remaining Missing Diabetes Labels:",
    df_ml[target_column]
    .isnull()
    .sum()
)

Remaining Missing Diabetes Labels: 0


# Compare Categorical Imputation Methods

Different techniques can assign different meanings to the same missing records.

Mode and Most Frequent usually produce the same fixed category, while Machine Learning-Based Imputation can produce record-specific predictions.

In [24]:
# Compare Diabetes imputation methods
diabetes_comparison = pd.DataFrame({
    "Original Missing": (
        df_categorical.loc[
            diabetes_missing_indexes,
            "diabetes_label"
        ]
    ),
    "Mode Imputation": (
        df_mode.loc[
            diabetes_missing_indexes,
            "diabetes_label"
        ]
    ),
    "Most Frequent": (
        df_most_frequent.loc[
            diabetes_missing_indexes,
            "diabetes_label"
        ]
    ),
    "ML-Based Imputation": (
        df_ml.loc[
            diabetes_missing_indexes,
            "diabetes_label"
        ]
    )
})

diabetes_comparison

,Original Missing,Mode Imputation,Most Frequent,ML-Based Imputation
5,NaN,No,No,No
10,NaN,No,No,No
15,NaN,No,No,No
20,NaN,No,No,Yes
25,NaN,No,No,No


In [25]:
# Compare category counts before and after Mode Imputation
distribution_comparison = pd.DataFrame({
    "Before Imputation": (
        df_categorical[
            "diabetes_label"
        ].value_counts()
    ),
    "After Mode Imputation": (
        df_mode[
            "diabetes_label"
        ].value_counts()
    ),
    "After ML Imputation": (
        df_ml[
            "diabetes_label"
        ].value_counts()
    )
}).fillna(0)

distribution_comparison

,Before Imputation,After Mode Imputation,After ML Imputation
diabetes_label,,,
No,170,175,174
Yes,124,124,125


In [26]:
# Verify all relevant methods
verification = pd.DataFrame({
    "Method": [
        "Mode",
        "New Missing Category",
        "Most Frequent",
        "ML-Based"
    ],
    "Remaining Missing Values": [
        df_mode[
            "diabetes_label"
        ].isnull().sum(),

        df_new_category[
            "sex_label"
        ].isnull().sum(),

        df_most_frequent[
            "diabetes_label"
        ].isnull().sum(),

        df_ml[
            "diabetes_label"
        ].isnull().sum()
    ]
})

verification

,Method,Remaining Missing Values
0,Mode,0
1,New Missing Category,0
2,Most Frequent,0
3,ML-Based,0


In [27]:
print("Original Dataset Shape:", df.shape)

print(
    "Original Dataset Missing Values:",
    df.isnull().sum().sum()
)

print(
    "Original Columns:",
    df.columns.tolist()
)

Original Dataset Shape: (299, 10)
Original Dataset Missing Values: 0
Original Columns: ['age', 'anaemia', 'creatinine_phosphokinase', 'diabetes', 'ejection_fraction', 'high_blood_pressure', 'platelets', 'serum_creatinine', 'serum_sodium', 'sex']


# Summary

In this notebook, we:

- Preserved the original Heart Failure dataset
- Converted binary features into readable categorical labels
- Introduced categorical missing values in a practice copy
- Applied Mode Imputation
- Created a new `"Missing"` category
- Applied Most Frequent Imputation using `SimpleImputer`
- Trained a Random Forest classifier for ML-Based Imputation
- Compared the imputed category values
- Compared category distributions
- Verified that the original dataset remained unchanged

## Key Learnings

- Mode fills missing data using the most common category.
- A new Missing category preserves missingness information.
- `SimpleImputer` supports pipeline-friendly Most Frequent Imputation.
- ML-Based Imputation predicts categories using related features.
- Mode and Most Frequent methods usually provide the same fixed value.
- Imputation can change category distributions.
- The selected technique must be justified and validated.

## Next Topic

Outlier Detection and Treatment